# VL09 - The Transformer architecture (MiniGPT)
In this seminar, we will implement transformer-based language model "MiniGPT". This MiniGPT is based on a Decoder Transformer block, causual self-attention, and a language modeling head. 

We will go over how to prepare data for the training, how to define the self-attention component, the transformer block and the language model head.

In [ ]:
import math
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, load_from_disk
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Prepare dataset 

We come back to our WikiText-2-raw-v1 dataset, and use it to train our small model.

We keep the tokenizer extremely simple:
- split on whitespace and punctuation, keeping puncutation so that the language model learns language structure
- build a small vocab of the most frequent tokens
This keeps training reasonably *fast* for our seminar.

We then transform the dataset to a long stream of tokens. No need to learn by sentences or even documents. We fit tokens in the context windows, consuming them in chunks.

### 1.1. Download and load dataset
Small dataset, 2M tokens.
````bash
$ python scripts/download_dataset.py "Salesforce/wikitext" "wikitext-2-raw-v1" data/wikitext-103-raw-v1
````

Large dataset, 103M tokens.
````bash
$ python scripts/download_dataset.py "Salesforce/wikitext" "wikitext-103-raw-v1" data/wikitext-103-raw-v1
````

In [ ]:
ds = load_from_disk("../../data/wikitext-2-raw-v1")
train_raw = ds["train"]["text"]
eval_raw = ds["validation"]["text"]
ds

### 1.2 Turning datasets into a long stream of tokens
Instead of splitting the dataset into sentences, we take the whole dataset and consume it as a stream of tokens. Punctuations are part of the tokens so the transformer architecture should be able to learn (given enough data) language structure, context switching directly from the data.

In [ ]:
import re
import spacy

TOKEN_RE = re.compile(
    r"[a-zA-Z]+(?:'[a-zA-Z]+)?|"  # words with optional apostrophe (don't, i'm)
    r"[.,!?;:()\-]"               # punctuation tokens as standalone
)

def clean_wikitext(lines):
    """Remove obvious wikitext artifacts."""
    for t in lines:
        t = t.strip()
        if not t:
            continue
        if re.match(r"^=+\s.*\s=+$", t):       # headings
            continue
        t = re.sub(r"\{\{.*?\}\}", "", t)      # {{ ... }}
        t = re.sub(r"\[\[|\]\]", "", t)        # [[ ]]
        yield t

def simple_tokenize(text):
    # find all words + punctuation, as separate tokens
    return TOKEN_RE.findall(text.lower())

def to_stream(cleaned_lines):
    stream = []
    for line in cleaned_lines:
        stream.extend(simple_tokenize(line))
    return stream

to_stream(["In the end, i don't know what to do!", "All is done"])


In [ ]:
# We transform the train and eval datasets into to a long stream of tokens
train_stream = to_stream(clean_wikitext(train_raw))
eval_stream  = to_stream(clean_wikitext(eval_raw))

len(train_stream), len(eval_stream)

### 1.3. Build the vocabulary
We build the vocabulary with the most common tokens in the training token stream. We limit this to speed up training and avoid the long tail of the count distribution.

In [ ]:
# Let's build the vocabulary
vocab_counts = Counter(train_stream)
most_common = vocab_counts.most_common(15000)  # or 10000
itos = ["<pad>", "<unk>"] + [w for w,_ in most_common]
stoi = {w:i for i,w in enumerate(itos)}
vocab_size = len(itos)

vocab_size

In [ ]:
most_common[1:50]

### 1.4. Encode the streams into array of vocab indexes
With the vocabulary built - and having an index for each token in our vocab - we encode our stream of tokens into a stream of token ids.

In [ ]:

def encode_stream(stream, stoi):
    """Map each token to an integer id."""
    stream_ids = [stoi.get(tok, stoi["<unk>"]) for tok in stream]
    return stream_ids

train_id_stream = encode_stream(train_stream, stoi)
eval_id_stream  = encode_stream(eval_stream,  stoi)

# to inspect an example:
print (encode_stream(["the", "game"], stoi))


len(train_id_stream), len(eval_id_stream)



**Quick check**: How many unks do we have in our encoded training dataset?

In [ ]:
unk_index = stoi["<unk>"]

total_tokens = len(train_id_stream)
unk_tokens = sum(1 for t in train_id_stream if t == unk_index)

unk_ratio = unk_tokens / total_tokens
print("Training - N tokens: ", total_tokens)
print("Training - N <unk>", unk_tokens)
print("Training - <unk> ratio", unk_ratio)

### 1.5 Converting the encoded token stream into context windows
Our decoder-only Transformer does not learn from sentences or documents.
Instead, it learns by predicting the next token inside a **fixed-size context window**  
(e.g., 64 tokens). This window is the model’s *entire world* during training.

To prepare the data in this format, we take our long, continuous stream of
token IDs and slice it into **non-overlapping chunks** of length `seq_len`.

In [ ]:
seq_len = 64
pad_idx = stoi["<pad>"]

def stream_to_windows(stream_ids, seq_len, pad_idx):
    """Create fixed-length, non-overlapping training blocks."""
    sequences = []
    for i in range(0, len(stream_ids), seq_len):
        chunk = stream_ids[i:i+seq_len]
        if len(chunk) < seq_len:
            chunk = chunk + [pad_idx] * (seq_len - len(chunk))
        sequences.append(torch.tensor(chunk, dtype=torch.long))
    return sequences

In [ ]:
train_seqs = stream_to_windows(train_id_stream, seq_len, pad_idx)
eval_seqs  = stream_to_windows(eval_id_stream,  seq_len, pad_idx)


len(train_seqs), len(eval_seqs)

## 2. Mini GPT model
In this section we build a very small **decoder-only Transformer**, following the same architectural ideas as GPT-style language models.  
The full model has three components:

1. **Causal Self-Attention** – the mechanism that lets the model attend only to past tokens.  
2. **A Decoder Transformer Block** – attention + feed-forward network + residual connections + layer norm.  
3. **Mini GPT** – token embedding -> transformer blocks -> linear LM head for next-token prediction.

Each piece is intentionally small so that we can train it live during the seminar.


### 2.1 Causal Self-Attention
Self-attention lets each token look at other tokens in the same context window. For a language model, this attention must be **causal**:  
each token can only attend to *previous* positions, never future ones.

Practically, this module:

- projects the input to queries, keys, and values  
- computes dot-product attention  
- applies a causal mask (upper triangular mask)  
- combines the heads  
- returns a transformed representation with the same dimensionality

This gives the model a way to blend information from earlier tokens.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, pad_mask=None):
        B, T, C = x.shape
    
        q, k, v = self.qkv(x).chunk(3, dim=-1)
    
        q = q.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
    
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
    
        # Causal mask (future tokens blocked)
        causal = torch.tril(torch.ones(T, T, device=x.device)).bool()
        att = att.masked_fill(~causal, float('-inf'))

        # This is so we do not compute attention to paddings! -inf is excluded
        if pad_mask is not None:
            # pad_mask: (B, T), True where pad
            pad_mask = pad_mask.unsqueeze(1).unsqueeze(1)   # (B,1,1,T)
            att = att.masked_fill(pad_mask, float('-inf'))
    
        att = F.softmax(att, dim=-1)
        y = att @ v
    
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)
    


### 2.2 Decoder Transformer Block
A decoder block stacks two main sub-layers:

1. **Causal Self-Attention**  
2. **Feed-Forward Network (MLP)**

Both sub-layers use **residual connections** and **layer normalization**, which help the model stabilize learning.

Practically, this module:

- runs causal attention  
- adds a residual connection + layer normalization  
- applies a small feed-forward network  
- adds another residual connection + normalization

This block is the main computational building unit of GPT-style models.


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model=256, num_heads=4, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.att = CausalSelfAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

        # <-- Dropout used exactly like in GPT-style implementations
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):

        # Residual connection 1 (self-attention)
        att_out = self.att(x, pad_mask=pad_mask)
        x = x + self.dropout(att_out)
        x = self.norm1(x)

        # Residual connection 2 (feed-forward network)
        ff_out = self.ff(x)
        x = x + self.dropout(ff_out)
        x = self.norm2(x)

        return x


### 2.3 Mini GPT (LM Head)
A GPT-style model takes token IDs, turns them into embedding vectors, processes them through several decoder blocks, and then predicts the next token at every position.

Practically, our Mini GPT model includes:

- a token embedding layer  
- one or more decoder blocks  
- a final linear projection (“LM head”) mapping hidden states to vocabulary logits

The LM head outputs a *probability distribution* over the next token.  
Training this model with cross-entropy on next-token prediction gives us a working, miniature GPT.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_layers=2, max_len=48, hidden_dim=512, num_heads=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model=d_model, num_heads=num_heads, hidden_dim=hidden_dim)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
    
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(pos)
    
        # Build pad mask
        pad_mask = (idx == pad_idx)
    
        for block in self.blocks:
            x = block(x, pad_mask=pad_mask)
    
        x = self.ln_f(x)
        return self.lm_head(x)



## 3. Training

We train our Mini GPT model using the standard **next-token prediction** objective.
For each window of tokens, the model sees positions `0 … T-2` as input and learns
to predict tokens `1 … T-1`.  
Training proceeds in mini-batches and we track both training and validation loss.

### 3.1 Training loop

The training loop repeatedly:
1. forms batches of token windows  
2. feeds inputs into the model  
3. computes next-token prediction loss  
4. performs a gradient update  
5. reports training and validation loss per epoch  

Practically, the code below:
- uses Adam for optimization  
- ignores `<pad>` tokens in the loss  
- prints deltas (Δ) to monitor convergence  
- stores the history for later visualization

In [ ]:
from tqdm.notebook import tqdm
     
def train_lm(model, train_seqs, val_seqs,
             batch_size=32, num_epochs=30, lr=8e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    pad_idx = stoi["<pad>"]
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

    history = []

    prev_train = None
    prev_val = None

    for epoch in range(1, num_epochs+1):
        random.shuffle(train_seqs)
        total_loss = 0.0
        num_batches = 0

        for i in tqdm(range(0, len(train_seqs), batch_size), desc=f"Epoch {epoch}"):
            batch = train_seqs[i:i+batch_size]
            if len(batch) < batch_size:
                continue

            batch = torch.stack(batch).to(device)  # (B, T)
            inputs  = batch[:, :-1]
            targets = batch[:, 1:]

            logits = model(inputs)
            loss = criterion(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        train_loss = total_loss / num_batches
        # We evaluate the loss on the evaluation dataset to test generalization
        val_loss = evaluate_lm_loss(model, criterion, val_seqs, batch_size=batch_size)
        history.append((train_loss, val_loss))

        if prev_train is None:
            print(f"Epoch {epoch:2d}/{num_epochs} — train={train_loss:.4f}  val={val_loss:.4f}")
        else:
            d_train = train_loss - prev_train
            d_val   = val_loss - prev_val
            print(
                f"Epoch {epoch:2d}/{num_epochs} — "
                f"train={train_loss:.4f} (Δ {d_train:+.4f})  "
                f"val={val_loss:.4f} (Δ {d_val:+.4f})"
            )

        prev_train = train_loss
        prev_val = val_loss

    return history


### 3.2 Evaluating loss on the validation dataset
 
We measure how well the model generalizes by computing the same next-token loss on held-out data *without* updating weights.
  
The function below:
- switches the model to evaluation mode  
- iterates over validation windows  
- computes average cross-entropy loss  
- returns the scalar value used for monitoring training

In [ ]:
def evaluate_lm_loss(model, criterion, val_seqs, batch_size=32):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for i in range(0, len(val_seqs), batch_size):
            batch = val_seqs[i:i+batch_size]
            if len(batch) < batch_size:
                continue

            batch = torch.stack(batch).to(device)  # (B, T)
            inputs  = batch[:, :-1]   # (B, T-1)
            targets = batch[:, 1:]    # (B, T-1)

            logits = model(inputs)    # (B, T-1, V)
            loss = criterion(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )

            total_loss += loss.item()
            num_batches += 1

    model.train()
    return total_loss / num_batches


### 3.3 Initialising components and model parameters
 
We select:
- hidden size (`d_model`)  
- number of layers  
- number of attention heads  
- feed-forward dimension  
- context window length (`max_len`)  

These hyperparameters define the capacity of the model.
  
Here, we instantiate our Mini GPT with reasonable dimensions for fast training.


In [ ]:
d_model = 192
hidden_dim = 384
num_layers = 2
num_heads = 4
max_len   = seq_len

model = MiniGPT(vocab_size, d_model=d_model, num_layers=num_layers,
                max_len=max_len, hidden_dim=hidden_dim, num_heads=num_heads).to(device)
model


### 3.4 Performing the training

We now train the model for a small number of epochs to observe learning behaviour
and track improvements in validation loss.

In [ ]:
history = train_lm(model, train_seqs, eval_seqs,
                   batch_size=32, num_epochs=5, lr=5e-4)


### 3.5 (Optional) Save trained weights
We can optionally save the model to reuse later.



In [ ]:
import torch

def save_checkpoint(model, stoi, itos, path="minigpt_checkpoint.pt"):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "stoi": stoi,
        "itos": itos,
        "config": {
            "vocab_size": model.token_emb.num_embeddings,
            "d_model": model.token_emb.embedding_dim,
            "num_layers": len(model.blocks),
            "max_len": model.pos_emb.num_embeddings,
            "hidden_dim": model.blocks[0].ff[0].out_features,
            "num_heads": model.blocks[0].att.num_heads
        }
    }
    torch.save(checkpoint, path)
    print(f"Saved checkpoint to: {path}")

# Example usage:
save_checkpoint(model, stoi, itos, path="minigpt_epoch30_lg_stream15_punkt.pt")


### 3.5 Loading the pretrained model 
We can directly load the model previously saved with the above function.

In [ ]:
def load_checkpoint(path="minigpt_checkpoint.pt"):
    checkpoint = torch.load(path, map_location=device)

    config = checkpoint["config"]
    stoi = checkpoint["stoi"]
    itos = checkpoint["itos"]

    # Recreate the model
    model = MiniGPT(
        vocab_size=config["vocab_size"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        max_len=config["max_len"],
        hidden_dim=config["hidden_dim"],
        num_heads=config["num_heads"]
    ).to(device)

    # Load weights
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded model from: {path}")
    return model, stoi, itos

# Example usage:
loaded_model, loaded_stoi, loaded_itos = load_checkpoint("minigpt_epoch30.pt")


## 4. Testing our model

Once the model is trained, we can test it by **sampling text** from it.  
This is the standard way to evaluate a decoder-only (GPT-style) model:  we provide a *prompt*, let the model predict the next token, append it, and repeat the process autoregressively.

This section implements a simple sampling function and tests the model on a prompt.


### 4.1 Sampling (autoregressive generation)

- We feed the model a prompt (a sequence of token IDs).  
- The model predicts the distribution of the next token.  
- We sample from this distribution (optionally using temperature).  
- We append the sampled token to the sequence and continue.
 
The functions below:
- handles tokenization and detokenization  
- runs the autoregressive loop  
- supports temperature for controlling randomness  
- ensures we keep only the last `seq_len` tokens when generating


In [ ]:
def decode_ids(ids):
    return [itos[i] for i in ids]

def encode_tokens(tokens):
    return torch.tensor([stoi.get(t, stoi["<unk>"]) for t in tokens], dtype=torch.long)


In [ ]:
def generate_greedy(model, prompt_tokens, max_new_tokens=30):
    model.eval()
    idx = encode_tokens(prompt_tokens).unsqueeze(0).to(device)  # (1, T)

    for _ in range(max_new_tokens):
        logits = model(idx)           # (1, T, V)
        next_logits = logits[:, -1, :]  # (1, V)
        next_id = next_logits.argmax(-1, keepdim=True)  # (1, 1)
        idx = torch.cat([idx, next_id], dim=1)

    return decode_ids(idx[0].tolist())

def generate_sample(model, prompt, max_new_tokens=60, temperature=1.0):
    model.eval()

    # tokenize prompt
    tokens = [stoi.get(tok, stoi["<unk>"]) for tok in simple_tokenize(prompt)]
    tokens = tokens[:seq_len]  # truncate if prompt is too long

    for _ in range(max_new_tokens):
        # get last seq_len tokens
        x = torch.tensor(tokens[-seq_len:], dtype=torch.long).unsqueeze(0).to(device)

        # forward pass: predict next token
        logits = model(x)  # (1, T, V)
        next_logits = logits[:, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)

        # sample from the distribution
        next_id = torch.multinomial(probs, num_samples=1).item()

        tokens.append(next_id)

    # detokenize
    result = [itos[t] for t in tokens]
    return result


### 4.2 Generating text from our model

Now we test generation using a short *prompt*.  
Because our model is small, generation will not be perfect,  
but it should show coherent structure, punctuation, and topic flow.

In [ ]:
prompt = "According to the game"
#generated = generate_greedy(model, prompt, max_new_tokens=60)
#print(" ".join(generated))


generated = generate_sample(model, prompt, max_new_tokens=64, temperature=.8)
print(" ".join(generated))

## 5. Reflection